# Machine-Checked Alignment Tax Theorem — Running Lean 4 in Colab

This notebook **actually runs** the Lean 4 formalization of the Alignment Tax Theorem in Google Colab. No local install required.

At the end, you'll have built the `nucleus` repository's portcullis-core Lean library and seen the proof of `alignmentTaxH1_eq_operational` (modulo one structural axiom) type-checked by Lean's kernel.

**Estimated runtime**: ~5-10 minutes (Mathlib cache download dominates).

**Open in Colab**: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/coproduct-opensource/nucleus/blob/main/notebooks/lean_in_colab.ipynb)

## Step 1: Install elan (Lean version manager)

`elan` is the Lean equivalent of rustup — it installs and manages Lean toolchains. About 30 seconds.

In [ ]:
!curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain none
import os
os.environ['PATH'] = f"{os.path.expanduser('~/.elan/bin')}:{os.environ['PATH']}"
!elan --version

## Step 2: Clone the nucleus repository

In [ ]:
!git clone --depth 1 https://github.com/coproduct-opensource/nucleus.git
%cd nucleus/crates/portcullis-core/lean

## Step 3: Fetch cached Mathlib oleans

Mathlib takes ~30 minutes to build from scratch. `lake exe cache get` downloads pre-built artifacts from the Mathlib release channel (~2-5 minutes on Colab's network).

This also installs the Lean 4.28.0-rc1 toolchain automatically (via `lean-toolchain` file).

In [ ]:
# lake automatically downloads Lean toolchain on first invocation
!lake exe cache get

## Step 4: Build the main theorem theorem's host file

Builds `AlignmentTaxBridge.lean` which contains `alignmentTaxH1_eq_operational` — the Alignment Tax Theorem.

Since Mathlib oleans are cached, only our own files compile (~30 seconds).

In [ ]:
!lake build AlignmentTaxBridge

## Step 5: Inspect the theorem

Let's print the actual Lean statement of the Alignment Tax Theorem:

In [ ]:
!grep -A 8 'alignmentTaxH1_eq_operational' AlignmentTaxBridge.lean | head -15

## Step 6: Verify sorry count

The theorem depends on one structural sorry (`gaussRankBool_eq_matrix_rank` — Gaussian elimination correctness over GF(2)). Everything else is proved.

In [ ]:
!echo '=== Sorries in the main theorem chain ==='
!grep -rn 'sorry' AlignmentTaxBridge.lean RankNullity.lean MatrixBridge.lean | grep -v '^--' | grep -v '//'

## What's been verified

After running all cells above, Lean's kernel has type-checked:

1. **`realising_set_size_ge_h1`** — every declassification set that kills H¹ has size ≥ rank H¹.
2. **`operationalAlignmentTaxH1_ge`** — operational tax ≥ rank H¹ (lower bound).
3. **`operationalAlignmentTaxH1_le`** — operational tax ≤ rank H¹ (upper bound).
4. **`alignmentTaxH1_eq_operational`** — the Alignment Tax Theorem.

All contingent on a single structural axiom: `gaussRankBool_eq_matrix_rank`, the classical correctness statement for Gaussian elimination over GF(2).

**That axiom is the only remaining gap between the Alignment Tax Theorem and unconditional formal verification.**

## Further exploration

Try building other targets:
- `lake build UniversalDetection` — information-theoretic detection impossibility (zero sorry)
- `lake build ComparisonTheorem` — full Čech cohomology framework (~15 min; native_decide heavy)
- `lake build MultiAgentCohomology` — multi-agent extension

Edit the Lean files and rebuild to explore the formalization interactively.